# Validation against the reference runs

Compares `gyaradax` runs with the reference runs of [gerkone/gyaradax-tests](https://huggingface.co/datasets/gerkone/gyaradax-tests):

- the test cases (`nonlinear/`, `linear/`, `kinetic/`), whose JSON diagnostics cover the whole GKW run,
- the long-horizon references (`long_horizon/`), with a nonlinear and a linear run per case.

Only the JSON diagnostics and the configs are needed, about 40 MB. Run `gyaradax` on the configs first, one output directory per case under `RUNS`:

```bash
ROOT=~/.cache/gyaradax/test_data RUNS=runs
hf download gerkone/gyaradax-tests --repo-type dataset --local-dir $ROOT --include "*.json" "*.yaml" "*input*.dat" "*.md"
# a test case, from its directory, from scratch, for the length of the GKW run ("steps" in its JSON)
(cd $ROOT/nonlinear/itg_weak_drive && gyaradax run config.yaml --from-scratch --n-steps 32000 --output-dir $OLDPWD/$RUNS/nonlinear/itg_weak_drive)
# a long-horizon case
gyaradax run $ROOT/long_horizon/kinetic_itg_mid_flux/config_nonlinear.yaml --output-dir $RUNS/long_horizon/kinetic_itg_mid_flux/nonlinear
gyaradax run $ROOT/long_horizon/kinetic_itg_mid_flux/config_linear.yaml --output-dir $RUNS/long_horizon/kinetic_itg_mid_flux/linear
```

Cases without a run are skipped. Turbulent runs separate from the reference within tens of $R/v_{th}$, so the comparison is statistical: time-averaged fluxes and spectra over the reference window, and the converged linear growth rates.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

sys.path.append("..")
from validation_stats import (
    aggregate_results,
    compare_config,
    compare_linear,
    load_reference,
    load_run,
    reference_cases,
    results_to_markdown,
    run_window,
)

ROOT = os.path.expanduser(os.environ.get("GYARADAX_TEST_DATA") or "~/.cache/gyaradax/test_data")
RUNS = os.environ.get("GYARADAX_VALIDATION_RUNS", os.path.join("..", "runs"))
if not os.path.exists(os.path.join(ROOT, "manifest.json")):
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="gerkone/gyaradax-tests",
        repo_type="dataset",
        local_dir=ROOT,
        allow_patterns=["*.json", "*.yaml", "*input*.dat", "*.md"],
    )

In [ ]:
def run_dirs(label):
    kind = label.split("/")[0]
    base = os.path.join(RUNS, label)
    if kind == "long_horizon":
        return os.path.join(base, "nonlinear"), os.path.join(base, "linear")
    return (None, base) if kind == "linear" else (base, None)


def has(d, f):
    return d if d and os.path.exists(os.path.join(d, f)) else None


cases = []
for label, case_dir in reference_cases(ROOT):
    nl_dir, lin_dir = run_dirs(label)
    cases.append(
        {
            "label": label,
            "nl_ref": load_reference(case_dir, "nonlinear"),
            "lin_ref": load_reference(case_dir, "linear"),
            "nl_dir": has(nl_dir, "fluxes.npz"),
            "lin_dir": has(lin_dir, "growth.npz"),
        }
    )
for c in cases:
    nl = "-" if c["nl_ref"] is None else ("run" if c["nl_dir"] else "no run")
    lin = "-" if c["lin_ref"] is None else ("run" if c["lin_dir"] else "no run")
    print(f"{c['label']:42s} nonlinear: {nl:7s} linear: {lin}")

## 1. Fluxes

Heat flux of each species, reference in black. The shaded band is the reference averaging window.

In [ ]:
nl_cases = [c for c in cases if c["nl_ref"] and c["nl_dir"]]
for c in nl_cases:
    ref, run = c["nl_ref"], load_run(c["nl_dir"])
    species = ref["species"]
    fig, axes = plt.subplots(1, len(species), figsize=(6 * len(species), 3), squeeze=False)
    for i, sp in enumerate(species):
        ax = axes[0, i]
        ax.plot(ref["time"], ref["fluxes"][sp]["heat"], color="k", lw=0.8, label="reference")
        ax.plot(run["time"], run["fluxes"][:, i, 1], lw=0.8, label="gyaradax")
        ax.axvspan(*ref["average"]["window"], color="0.9", zorder=-1)
        ax.set(xlabel=r"$t$ [$R/v_{th}$]", ylabel=f"{sp} heat flux", title=c["label"])
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 2. Spectra

$k_y$ and $k_x$ spectra of $|\phi|^2$, averaged over the reference window (or the same fraction at the end of a shorter run).

In [ ]:
for c in nl_cases:
    ref, run = c["nl_ref"], load_run(c["nl_dir"])
    if "ky_spec" not in run:
        continue
    rt = np.asarray(ref["time"])
    t0, t1 = ref["average"]["window"]
    rmask = (rt >= t0) & (rt <= t1)
    s0, s1 = run_window(run["time"], ref)
    smask = (run["time"] >= s0) & (run["time"] <= s1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    for ax, key, rkey, k in ((axes[0], "ky_spec", "ky_spectrum", "ky"), (axes[1], "kx_spec", "kx_spectrum", "kx")):
        kk = np.asarray(ref[k])
        ax.semilogy(kk, np.asarray(ref[rkey])[rmask].mean(axis=0), "k.-", ms=3, label="reference")
        if run[key].shape[1] == len(kk):
            ax.semilogy(kk, run[key][smask].mean(axis=0), ".-", ms=3, label="gyaradax")
        ax.set(xlabel=f"${k[0]}_{k[1]}$", ylabel=r"$\langle|\phi|^2\rangle$", title=c["label"])
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 3. Linear growth rates

Growth rate at the end of each linear run against the converged reference. The relative difference is reported over the $k_y$ with $\gamma$ above a quarter of the maximum.

In [ ]:
for c in [c for c in cases if c["lin_ref"] and c["lin_dir"]]:
    res = compare_linear(c["lin_dir"], c["lin_ref"])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(res["ky"], res["ref_gamma"], "k.-", label="reference")
    ax.plot(res["ky"], res["sim_gamma"], ".-", label=f"gyaradax, t = {res['t_end']:.0f}")
    ax.axhline(0, color="0.6", lw=0.5)
    ax.set(xlabel=r"$k_y$", ylabel=r"$\gamma$ [$v_{th}/R$]", title=c["label"])
    ax.legend(fontsize=8)
    plt.show()
    print(f"{c['label']}: max relative difference over the unstable ky {res['max_rel_diff']:.2e}")

## 4. Statistical validation

Quantitative comparison of the ion channel of every nonlinear run with a reference:

- **Spectra**: KS test, Pearson correlation, relative $L_2$ error on the time-averaged $k_x$ and $k_y$ spectra
- **Growth rates**: mean $\pm$ std of the time-averaged growth rate profile
- **Flux averages**: mean $\pm$ autocorrelation-corrected standard error, and the $z$ score of the difference

In [ ]:
all_stats = {c["label"]: compare_config([c["nl_dir"]], c["nl_ref"], species=0) for c in nl_cases}
aggregated = aggregate_results(all_stats)
if aggregated:
    display(Markdown(results_to_markdown(aggregated, per_config=all_stats)))
else:
    print(f"no nonlinear runs under {RUNS}")

### Interpretation

- **KS test**: Low statistic ($\lesssim 0.1$) indicates consistent spectral shapes. Note: discrete k-grids limit the test's resolution.
- **Pearson $r$**: Values close to 1.0 confirm spectral shape agreement.
- **Relative $L_2$**: Amplitude-weighted distance in log space; values $\lesssim 0.2$ indicate good quantitative match.
- **Growth rate**: Mean $\pm$ std over the window. Agreement is assessed by overlap of error bars.
- **Flux $z$ score**: $|\mu_\mathrm{sim} - \mu_\mathrm{ref}|$ over the combined standard error; values below 2 are consistent.